# Initialize Unity Catalog Schema, Volume, and Tables

Lightweight setup task that creates the UC catalog, schema, volume, and
empty tables if they don't already exist. Delegates all table and function
creation to `Catalog.init_tables()`, which reads the `.sql` scripts in
`src/dbx/pixels/resources/sql/`.

This notebook is self-contained — requires only `dbx.pixels`.

In [ ]:
%pip install fsspec --quiet

# Add src/ to path so dbx.pixels is importable from install/
import sys, os
_nb_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_nb_dir = "/Workspace" + os.path.dirname(_nb_ctx.notebookPath().get())
_repo_root = os.path.normpath(os.path.join(_nb_dir, ".."))
sys.path.insert(0, os.path.join(_repo_root, "src"))

In [ ]:
dbutils.widgets.text("table", "main.pixels_solacc.object_catalog", "UC Table")
dbutils.widgets.text("volume", "main.pixels_solacc.pixels_volume", "UC Volume")

In [ ]:
# Job parameters (table, volume) are injected via dbutils widgets.
table = dbutils.widgets.get("table")
volume = dbutils.widgets.get("volume")
uc_schema = ".".join(table.split(".")[:2])

# Create schema and volume (Catalog.init_tables does not handle these)
spark.sql(f"CREATE DATABASE IF NOT EXISTS {uc_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {volume}")

# Delegate all table and function creation to the .sql scripts
from dbx.pixels import Catalog

catalog = Catalog(spark, table=table, volume=volume)
catalog.init_tables()

print(f"All UC objects created successfully. table={table}, volume={volume}")

In [ ]:
# DBTITLE 1,Upload Build Artifacts to Volume
import shutil, os, glob

_vol_parts = volume.split(".")
_volume_base = f"/Volumes/{_vol_parts[0]}/{_vol_parts[1]}/{_vol_parts[2]}"
_vol_dist_dir = os.path.join(_volume_base, "dist")
os.makedirs(_vol_dist_dir, exist_ok=True)

for _artifact in ["vista3d.tar.gz"]:
    _src = os.path.join(_repo_root, "dist", _artifact)
    _dst = os.path.join(_vol_dist_dir, _artifact)
    if os.path.isfile(_src):
        shutil.copy2(_src, _dst)
        print(f"  Uploaded {_artifact} ({os.path.getsize(_src)/1e6:.1f}MB) to {_dst}")
    else:
        print(f"  ⚠ {_artifact} not found at {_src} — run 'make build' first")

In [ ]:
# DBTITLE 1,Tag UC Assets
catalog_name, schema_name, _ = table.split(".")

_tag_targets = [
    f"ALTER SCHEMA {catalog_name}.{schema_name} SET TAGS ('accelerator' = 'pixels')",
    f"ALTER TABLE {catalog_name}.{schema_name}.object_catalog SET TAGS ('accelerator' = 'pixels')",
    f"ALTER TABLE {catalog_name}.{schema_name}.object_catalog_unzip SET TAGS ('accelerator' = 'pixels')",
    f"ALTER TABLE {catalog_name}.{schema_name}.object_catalog_autoseg_result SET TAGS ('accelerator' = 'pixels')",
    f"ALTER TABLE {catalog_name}.{schema_name}.object_catalog_redaction SET TAGS ('accelerator' = 'pixels')",
    f"ALTER TABLE {catalog_name}.{schema_name}.stow_operations SET TAGS ('accelerator' = 'pixels')",
]

for stmt in _tag_targets:
    try:
        spark.sql(stmt)
        _target = stmt.split("SET TAGS")[0].strip()
        print(f"  Tagged: {_target}")
    except Exception as e:
        print(f"  Skip: {str(e)[:80]}")